<div align="center">
  <a href="https://colab.research.google.com/github/PrunaAI/ai-efficiency-courses/blob/main/solutions/07-finetune_llm.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---
**💡 Tip**: Click the button above to open this notebook in Google Colab for free GPU access!

## Installation

This notebook includes automatic setup cells that will install the project from git repository with UV.

**Note**: Run the setup cells below before starting the exercises.

In [ ]:
# Install project directly from git repository
!uv pip install git+https://github.com/PrunaAI/ai-efficiency-courses.git

# 08: Finetune LLMs

Welcome to the next lecture of the LLM Efficiency course! 🎯

In this tutorial, we’ll explore how to finetune Large Language Models (LLMs) to improve their performance after applying some compression techniques. After quantization and pruning, we can achieve a significant reduction in the model size and inference time. However, this comes at the cost of accuracy. Finetuning the model can help us to recover the quality of the model.

> The content from the chapter 5 [slides](https://github.com/PrunaAI/ai-efficiency-courses/blob/main/slides/05-finetune_language_models.pdf) will help you to go through this notebook.

By the end of this lecture, you will:
- Understand how to evaluate baseline model quality before finetuning.
- Learn strategies for finetuning with in-distribution, out-of-distribution, and even random data.
- Explore how dataset size and distribution affects the efficiency.

Let’s  get started  on how to make your LLMs more adaptable and effective!


## 1. Imports

As we've already installed the project, we can import the necessary libraries. We will be using torch and transformers for this tutorial as interfaces to the model and tokenizer. On top of that, we will be using matplotlib for basic plotting. We recommend to checkout the [Pruna documentation](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/evaluate.html) for access to AI efficiency functions.

> Take into account that for this notebook, we will be using `pruna_pro` package. For more information, please check the [documentation](https://docs.pruna.ai/en/stable/docs_pruna_pro/user_manual/pruna_pro.html).

Beyond external libraries, this course comes with the course local package which contains a lot of utils that you can use in the notebooks.

Before starting, we highly recommend to check the basic [Hugging Face setup in the readme](https://github.com/PrunaAI/ai-efficiency-courses?tab=readme-ov-file#configuration) including updating cache directory, loging in to hugging face.

In [ ]:
import gc
import copy
import random

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

from pruna_pro import SmashConfig
from pruna_pro import smash
from pruna.data.pruna_datamodule import PrunaDataModule
from pruna.data.utils import split_train_into_train_val_test
from pruna.evaluation.evaluation_agent import EvaluationAgent
from pruna.evaluation.metrics import TorchMetricWrapper, LatencyMetric
from pruna.evaluation.task import Task

## 2. Utils

In this section, we'll leverage some course utilities to streamline our workflow.
These utilities will help us:
- Load and manage lists of model ids that we have verified to work.
- Generate informative plots for model analysis.
- Iterate efficiently over evaluation and model configuration options.

Let's first load our models. We will use `SMALL_MODEL_IDS`, which are sub 1B parameters which should be easy to download and load into memory. We recommend starting with these smaller models but feel free to experiment with other models until you reach your GPU memory limit!

In [ ]:
from course import SMALL_MODEL_IDS as MODEL_IDS
# from course import MEDIUM_MODEL_IDS as MODEL_IDS
# from course import LARGE_MODEL_IDS as MODEL_IDS

MODEL_IDS

['facebook/opt-125m',
 'facebook/opt-350m',
 'HuggingFaceTB/SmolLM-135M-instruct',
 'HuggingFaceTB/SmolLM2-135M-Instruct',
 'HuggingFaceTB/SmolLM-360M-Instruct',
 'HuggingFaceTB/SmolLM2-360M-Instruct',
 'PleIAs/Pleias-350m-Preview',
 'PleIAs/Pleias-Pico',
 'LiquidAI/LFM2-350M',
 'LiquidAI/LFM2-700M']

## 3. Finetune an LLM

> We recommend to check the [evaluation guide](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/evaluate.html) and [metrics overview](https://docs.pruna.ai/en/stable/reference/evaluation.html#metrics-overview) in the pruna documentation for the implementation details.

To get familiar with the evaluation process and the pruna library, implement a function that evaluates the models with the perplexity metric.

In [10]:
def smash_evaluate_perplexity_time(model, tokenizer, smash_config, dataset="WikiText"):
    ### To Complete ###
    model_copy = copy.deepcopy(model)

    if smash_config:
        model_copy = smash(model_copy, smash_config)
    metrics = [
        LatencyMetric(
            n_iterations=100,
            n_warmup_iterations=10,
            device="cuda",
            timing_type="sync",
        ),
        TorchMetricWrapper(metric_name="perplexity", call_type="y_gt"),
    ]
    task = Task(
        metrics, datamodule=PrunaDataModule.from_string(dataset, tokenizer=tokenizer)
    )
    eval_agent = EvaluationAgent(task)
    results = eval_agent.evaluate(model_copy)

    del model_copy
    torch.cuda.empty_cache()
    gc.collect()
    ### End of To Complete ###

    return results

### 3.1 Evaluate the Base Model Quality

In this section, you'll evaluate the base model so you can compare the results with the finetuned versions

**Why is this important?**
Evaluating a base model establishes a performance baseline. This is crucial to measure and compare the gainings after fine-tuning the model.

**Your tasks:**
- Compute the base model perplexity using the WikiText dataset.

**Key questions to answer as you explore:**
- What is the base model’s perplexity?
- Which are the trade-offs between perplexity and latency?

In [ ]:
### To Complete ###
model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0]).cuda()
tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS[0])

results = smash_evaluate_perplexity_time(model, tokenizer, None, dataset="WikiText")
print(results)
### End of To Complete ###

### 3.2 Finetune LLM quality with in-distribution data

In this section, you’ll explore how to finetune a quantized LLM using in-distribution data, i.e., data drawn from the same domain as your evaluation set.

**Why is this important?**
Quantization often reduces model quality due to compression, but finetuning can help recover lost performance. By adapting the model with data similar to the evaluation set, you’ll test whether quality improves without significantly affecting inference latency.

**Your tasks:**
- Finetune in-place or by adding parameters to a quantized LLM with Quanto
- Evaluate its quality and latency metric on the WikiText dataset.

**Key questions to answer as you explore:**
- Does finetuning improve perplexity compared to the baseline model?
- Do you observe any latency differences after finetuning? What are the potential reasons?

In [11]:
### To Complete ###
smash_config = SmashConfig()
smash_config.add_tokenizer(MODEL_IDS[0])
smash_config.add_data("WikiText", tokenizer=tokenizer)
smash_config["quantizer"] = "quanto"
smash_config["quanto_weight_bits"] = "qint4"
smash_config["recoverer"] = "text_to_text_inplace_perp"
# smash_config['recoverer'] = "text_to_text_perp"
model = model.to("cuda")

results = smash_evaluate_perplexity_time(
    model, tokenizer, smash_config, dataset="WikiText"
)
print(results)
### End of To Complete ###

INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f4955a23370>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=None)...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Verifying Pruna token.
INFO - You have used 345 hours this month.
INFO - Starting quantizer quanto...
INFO - quantizer quanto was applied successfully.
INFO - Star

Converting train dataset to ChatML:   0%|          | 0/17556 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/17556 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/17556 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17556 [00:00<?, ? examples/s]

Step,Training Loss
2194,3.955200
4388,3.829000
6582,3.804500
8776,3.747200
10970,3.727300
13164,3.734500
15358,3.711000
17552,3.691700


INFO - recoverer text_to_text_inplace_perp was applied successfully.
INFO - You have used 348 hours this month.
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f4955a23370>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=None)...
INFO - Using provided list of metric instances.
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- T

{'perplexity_y_gt': 29.485769271850586, 'inference_elapsed_time_ms_@1': 5949.783557891846, 'inference_latency_ms_@1': 59.497835578918455, 'inference_throughput_batches_per_ms_@1': 0.01680733408652473}


### 3.3 Finetune LLM quality with more/less in-distribution data

In this section, you’ll investigate how the distribution of the data influences the effectiveness of finetuning a quantized LLM.

**Why is this important?**
Finetuning with more representative in-distribution data should enhance performance recovery after quantization, whereas finetuning with less representative data might yield smaller gains. Understanding this helps you design the finetuning strategy when working with limited data sources.

**Your tasks:**
- Finetune in-place or by adding parameters to a quantized LLM with Quanto
- Evaluate its quality and latency metric on the WikiText dataset.

**Key questions to answer as you explore:**
- Does finetuning improve perplexity compared to the baseline model?
- How does the amount and distribution of data affect the finetuning process?
- Do you observe any latency differences after finetuning? What are the potential reasons?

In [12]:
### To Complete ###
train_ds, val_ds, test_ds = load_dataset(
    "mikasenghaas/wikitext-2", split=["train", "validation", "test"]
)
train_ds = train_ds.select(range(1000))

smash_config = SmashConfig()
smash_config.add_tokenizer(MODEL_IDS[0])
smash_config.add_data((train_ds, val_ds, test_ds), collate_fn="text_generation_collate")
smash_config["quantizer"] = "quanto"
smash_config["quanto_weight_bits"] = "qint4"
smash_config["recoverer"] = "text_to_text_inplace_perp"
# smash_config['recoverer'] = "text_to_text_perp"

results = smash_evaluate_perplexity_time(
    model, tokenizer, smash_config, dataset="WikiText"
)
print(results)
### End of To Complete ###

INFO - Using max_seq_len of tokenizer: None
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f4955a23370>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=None)...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Verifying Pruna token.
INFO - You have used 348 hours this month.
INFO - Starting quantizer quanto...
INFO - quantizer 

Converting train dataset to ChatML:   0%|          | 0/1000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss
125,4.301900
250,4.249300
375,4.003500
500,3.966100
625,3.875400
750,3.956700
875,3.815600
1000,3.809000


INFO - recoverer text_to_text_inplace_perp was applied successfully.
INFO - You have used 348 hours this month.
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f4955a23370>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=None)...
INFO - Using provided list of metric instances.
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- T

{'perplexity_y_gt': 35.11149215698242, 'inference_elapsed_time_ms_@1': 6988.028305053711, 'inference_latency_ms_@1': 69.88028305053712, 'inference_throughput_batches_per_ms_@1': 0.01431018817248929}


### 3.3 Finetune LLM quality with random data

In this section, you’ll test what happens when a quantized LLM is finetuned using random data instead of in-distribution data.

**Why is this important?**
Using random data introduces noise so it is expected to worsen the model. However, this serves as a useful baseline to contrast against finetuning with in-distribution or representative data.

**Your tasks:**
- Finetune in-place or by adding parameters to a quantized LLM with Quanto
- Evaluate its quality and latency metric on the WikiText dataset.

**Key questions to answer as you explore:**
- Does finetuning with random data improve, degrade, or leave perplexity unchanged?
- How does this baseline highlight the importance of data quality in post-quantization finetuning?
- Do you observe any latency differences after finetuning? What are the potential reasons?

In [11]:
### To Complete ###
from datasets import Dataset

dataset = Dataset.from_dict(
    {
        "text": [
            "".join([chr(random.randint(97, 122)) for _ in range(100)])
            for _ in range(1000)
        ]
    }
)
train_ds, val_ds, test_ds = split_train_into_train_val_test(dataset, seed=42)

model = model.to("cuda")
smash_config = SmashConfig()
smash_config.add_tokenizer(MODEL_IDS[0])
smash_config.add_data((train_ds, val_ds, test_ds), collate_fn="text_generation_collate")
smash_config["device"] = "cuda"
smash_config["quantizer"] = "quanto"
smash_config["quanto_weight_bits"] = "qint4"
smash_config["recoverer"] = "text_to_text_inplace_perp"
# smash_config['recoverer'] = "text_to_text_perp"

results = smash_evaluate_perplexity_time(
    model, tokenizer, smash_config, dataset="WikiText"
)
print(results)
### End of To Complete ###

INFO - Loaded only training, splitting train 80/10/10 into train, validation and test...
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f8538f52dd0>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=None)...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Verifying Pruna token.
INFO - You have used 359 hours this month.
INFO - S

Converting train dataset to ChatML:   0%|          | 0/800 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Step,Training Loss
100,6.376800
200,6.031500
300,5.823100
400,5.873100
500,5.816900
600,5.804300
700,5.812600
800,5.867800


INFO - recoverer text_to_text_inplace_perp was applied successfully.
INFO - You have used 360 hours this month.
INFO - Using call_type: y_gt for metric perplexity
INFO - Using max_seq_len of tokenizer: None
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f8538f52dd0>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=None)...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maxim

{'perplexity_y_gt': 167.92860412597656, 'inference_elapsed_time_ms_@1': 3896.4342079162598, 'inference_latency_ms_@1': 38.9643420791626, 'inference_throughput_batches_per_ms_@1': 0.025664490830316914}


### 3.3 Finetune LLM quality with out-of-distribution data

In this section, you’ll investigate how finetuning a quantized LLM with out-of-distribution data affects its performance and efficiency.

**Why is this important?**
Finetuning with out-of-distribution data can be challenging and result inunpredictable effects. It can sometimes hurt performance on the evaluation set due to domain mismatch, or in rare cases provide indirect benefits by improving generalization.

**Your tasks:**
- Finetune in-place or by adding parameters to a quantized LLM with Quanto
- Evaluate its quality and latency metric on the WikiText dataset.

**Key questions to answer as you explore:**
- Does finetuning improve perplexity compared to the baseline model?
- How does the amount and distribution of data affect the finetuning process?
- Do you observe any latency differences after finetuning? What are the potential reasons?

In [13]:
### To Complete ###
train_ds = load_dataset("SamuelYang/bookcorpus")["train"]
train_ds, val_ds, test_ds = split_train_into_train_val_test(train_ds, seed=42)
train_ds = train_ds.select(range(1000))

model = model.to("cuda")
smash_config = SmashConfig()
smash_config.add_tokenizer(MODEL_IDS[0])
smash_config.add_data((train_ds, val_ds, test_ds), collate_fn="text_generation_collate")
smash_config["device"] = "cuda"
smash_config["quantizer"] = "quanto"
smash_config["quanto_weight_bits"] = "qint4"
smash_config["recoverer"] = "text_to_text_inplace_perp"
# smash_config['recoverer'] = "text_to_text_perp"

results = smash_evaluate_perplexity_time(
    model, tokenizer, smash_config, dataset="WikiText"
)
print(results)
### End of To Complete ###

INFO - Loaded only training, splitting train 80/10/10 into train, validation and test...
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f8538f52dd0>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=None)...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Verifying Pruna token.
INFO - You have used 360 hours this month.
INFO - S

Converting train dataset to ChatML:   0%|          | 0/1000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss
125,5.263400
250,4.252200
375,4.094600
500,4.058200
625,4.063000
750,3.880100
875,3.865400
1000,3.761500


INFO - recoverer text_to_text_inplace_perp was applied successfully.
INFO - You have used 360 hours this month.
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f8538f52dd0>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=None)...
INFO - Using provided list of metric instances.
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- T

{'perplexity_y_gt': 70.56623840332031, 'inference_elapsed_time_ms_@1': 3814.258159637451, 'inference_latency_ms_@1': 38.14258159637451, 'inference_throughput_batches_per_ms_@1': 0.026217417860752535}


## Conclusion: What We've Learned About Fine-tuning LLMs

In this module, we explored how fine-tuning can help recover the quality of a quantized LLM and the data distribution influence the finetuning results. Here's a recap of the key concepts:

- **Data Quality Matters**: The quality of finetuning data is critical for post-quantization recovery.
- **Stay in-distribution**: Well-matched in-distribution data delivers the best balance of perplexity and latency.
- **Avoid mismatches**: Less aligned, out-of-distribution, or random data degradate quality, with random data being the most harmful.
- **Be careful**: Efficiency gains observed with random or out-of-distribution data reflect degraded modeling ability, not a proper optimization.


### Next Steps: Final Project

Now that you understand how to fine-tune and recover the quality of LLMs. Ypu're ready to go and start your final project.

## ⭐ Bonus Exercise: Combine Different Data Distributions

As a bonus, extend your analysis to explore how combining different data distributions can help, or not, improve the quality and generalization of the finetuned model.

**Your tasks:**

- Finetune a quantized LLM with Quanto or a different quantizer and with different data distributions.

**Questions:**
- What happens when you combine distributions—does performance improve across all tasks, or does one dominate?
- Does mixing diverse distributions help the model generalize better?
- What happens if one distribution is underrepresented?